In [1]:
import html
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
df = pd.read_csv("combined_courses.csv")

In [3]:
df.head()

,course_id,course_name,description,skills,subject,level,organization,provider,rating,reviews_count,...,lectures_count,duration,instructor,price,language,image_url,url,certificate_type,course_type,source_file
0,1,Vector Calculus for Engineers,This course covers both the basic theory and a...,"Mathematical Modeling, Physics, Engineering, S...",Math and Logic,Beginner Level,The Hong Kong University of Science and Techno...,Coursera,4.8,1356.0,...,NaN,Approx. 29 hours to complete,Jeffrey R. Chasnov,free,English,https://s3.amazonaws.com/coursera_assets/meta_...,https://www.coursera.org/learn/vector-calculus...,Shareable Certificate,course,webautomation_coursera.csv | processed_courser...
1,2,Fibonacci Numbers and the Golden Ratio,Learn the mathematics behind the Fibonacci num...,"Recreational Mathematics, Discrete Mathematics...",Math and Logic,Beginner Level,The Hong Kong University of Science and Techno...,Coursera,4.8,1156.0,...,NaN,Approx. 10 hours to complete,Jeffrey R. Chasnov,free,English,https://s3.amazonaws.com/coursera_assets/meta_...,https://www.coursera.org/learn/fibonacci,Shareable Certificate,course,webautomation_coursera.csv | processed_courser...
2,3,Matrix Algebra for Engineers,"This course is all about matrices, and concise...","Algebra, Advanced Mathematics, Statistical Mod...",Math and Logic,Beginner Level,The Hong Kong University of Science and Techno...,Coursera,4.9,4462.0,...,NaN,Approx. 20 hours to complete,Jeffrey R. Chasnov,free,English,https://s3.amazonaws.com/coursera_assets/meta_...,https://www.coursera.org/learn/matrix-algebra-...,Shareable Certificate,course,webautomation_coursera.csv | processed_courser...
3,4,Game Theory,"Popularized by movies such as ""A Beautiful Min...",NaN,Economics,Beginner Level,-,Coursera,4.6,NaN,...,NaN,Approx. 18 hours to complete,NaN,free,English,https://s3.amazonaws.com/coursera_assets/meta_...,https://www.coursera.org/learn/game-theory-1,Shareable Certificate,course,webautomation_coursera.csv
4,5,SQL for Data Science,As data collection has increased exponentially...,"Data Science, Query Languages, Data Integratio...",Data Analysis,Beginner Level,"University of California, Davis",Coursera,4.6,16609.0,...,NaN,Approx. 14 hours to complete,Sadie St. Lawrence,free,English,https://s3.amazonaws.com/coursera_assets/meta_...,https://www.coursera.org/learn/sql-for-data-sc...,Shareable Certificate,course,webautomation_coursera.csv | processed_courser...


In [4]:
df.shape

(24646, 21)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24646 entries, 0 to 24645
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   course_id          24646 non-null  int64  
 1   course_name        24646 non-null  object 
 2   description        15148 non-null  object 
 3   skills             12340 non-null  object 
 4   subject            8447 non-null   object 
 5   level              24605 non-null  object 
 6   organization       16341 non-null  object 
 7   provider           24646 non-null  object 
 8   rating             14206 non-null  float64
 9   reviews_count      22378 non-null  float64
 10  students_enrolled  11312 non-null  float64
 11  lectures_count     8273 non-null   float64
 12  duration           23088 non-null  object 
 13  instructor         20242 non-null  object 
 14  price              8563 non-null   object 
 15  language           2072 non-null   object 
 16  image_url          355

In [6]:
df.isnull().sum()

course_id                0
course_name              0
description           9498
skills               12306
subject              16199
level                   41
organization          8305
provider                 0
rating               10440
reviews_count         2268
students_enrolled    13334
lectures_count       16373
duration              1558
instructor            4404
price                16083
language             22574
image_url            21090
url                   5389
certificate_type     22842
course_type           4703
source_file              0
dtype: int64

In [7]:
# ==========================================================
# 1. LOAD DATASET
# ==========================================================
df = pd.read_csv("combined_courses.csv")
# Keep the original dataset unchanged
data = df.copy()
# ==========================================================
# 2. CONVERT EMPTY PLACEHOLDERS INTO REAL MISSING VALUES
# ==========================================================

# Some datasets use text such as "Not Found" or "-" instead of NaN.
missing_placeholders = [
    "",
    " ",
    "-",
    "--",
    "N/A",
    "n/a",
    "NA",
    "na",
    "None",
    "none",
    "NULL",
    "null",
    "Not Found",
    "not found",
    "Not Available",
    "not available"
]

object_columns = data.select_dtypes(include="object").columns

for column in object_columns:
    data[column] = (
        data[column]
        .replace(missing_placeholders, np.nan)
    )


# ==========================================================
# 3. RECORD WHICH VALUES WERE ORIGINALLY MISSING
# ==========================================================

# These indicator columns allow you to know whether a value
# was original or filled later.

original_columns = data.columns.tolist()

for column in original_columns:
    if data[column].isnull().any():
        data[f"{column}_was_missing"] = (
            data[column]
            .isnull()
            .astype("int8")
        )


# ==========================================================
# 4. NORMALIZE PROVIDER NAMES
# ==========================================================

# Your dataset contains both "Coursera" and "coursera".

provider_mapping = {
    "coursera": "Coursera",
    "udemy": "Udemy",
    "edx": "edX"
}

data["provider"] = (
    data["provider"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.casefold()
    .map(provider_mapping)
    .fillna("Unknown")
)


# ==========================================================
# 5. HANDLE NLP TEXT COLUMNS
# ==========================================================

# Empty strings are best for TF-IDF.
# Missing text should contribute no words to the NLP model.

nlp_text_columns = [
    "course_name",
    "description",
    "skills",
    "subject",
    "organization",
    "instructor"
]

for column in nlp_text_columns:
    data[column] = (
        data[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# Provider is also text, but it has already been normalized.
data["provider"] = (
    data["provider"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)


# ==========================================================
# 6. HANDLE CATEGORICAL COLUMNS
# ==========================================================

# For filters and app display, use "Unknown".

categorical_columns = [
    "level",
    "language",
    "certificate_type",
    "course_type"
]

for column in categorical_columns:
    data[column] = (
        data[column]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )


# ==========================================================
# 7. CREATE NLP-SAFE CATEGORICAL COLUMNS
# ==========================================================

# Do not put the word "Unknown" into TF-IDF.
# These separate columns use an empty string when the original
# value was missing.

data["nlp_level"] = (
    df["level"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data["nlp_language"] = (
    df["language"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data["nlp_certificate_type"] = (
    df["certificate_type"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data["nlp_course_type"] = (
    df["course_type"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ==========================================================
# 8. HANDLE RATING
# ==========================================================

# Ensure rating is numeric.
data["rating"] = pd.to_numeric(
    data["rating"],
    errors="coerce"
)

# Ratings outside 0–5 are treated as invalid.
data.loc[
    (data["rating"] < 0) | (data["rating"] > 5),
    "rating"
] = np.nan

# First fill using the median rating from the same provider.
provider_rating_median = (
    data.groupby("provider")["rating"]
    .transform("median")
)

# If the provider median is unavailable, use global median.
global_rating_median = data["rating"].median()

data["rating"] = (
    data["rating"]
    .fillna(provider_rating_median)
    .fillna(global_rating_median)
    .round(2)
)


# ==========================================================
# 9. HANDLE COUNT COLUMNS
# ==========================================================

count_columns = [
    "reviews_count",
    "students_enrolled",
    "lectures_count"
]

for column in count_columns:

    # Convert values to numeric
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

    # Negative counts are invalid
    data.loc[data[column] < 0, column] = np.nan

    # Calculate provider median
    provider_median = (
        data.groupby("provider")[column]
        .transform("median")
    )

    # Calculate overall median
    global_median = data[column].median()

    # Fill missing values
    data[column] = (
        data[column]
        .fillna(provider_median)
        .fillna(global_median)
        .round()
        .astype("int64")
    )


# ==========================================================
# 10. CREATE CONSERVATIVE POPULARITY COLUMNS
# ==========================================================

# Imputed counts are useful for analysis, but for popularity
# ranking it is safer to treat unavailable evidence as zero.

data["reviews_for_ranking"] = (
    pd.to_numeric(
        df["reviews_count"],
        errors="coerce"
    )
    .clip(lower=0)
    .fillna(0)
)

data["students_for_ranking"] = (
    pd.to_numeric(
        df["students_enrolled"],
        errors="coerce"
    )
    .clip(lower=0)
    .fillna(0)
)


# ==========================================================
# 11. HANDLE PRICE
# ==========================================================

# Price contains text such as "free", "Enroll", or currency
# values, so do not calculate a median yet.

data["price"] = (
    data["price"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Create a free-course indicator
data["is_free"] = (
    data["price"]
    .str.casefold()
    .str.contains(r"\bfree\b", regex=True, na=False)
    .astype("int8")
)


# ==========================================================
# 12. HANDLE DURATION
# ==========================================================

# Duration contains mixed units such as hours, weeks and months.

data["duration"] = (
    data["duration"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)


# ==========================================================
# 13. HANDLE IMAGE URL
# ==========================================================

# Use an empty string when no image is available.
# Flutter can display a local placeholder image.

data["image_url"] = (
    data["image_url"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data["has_image"] = (
    data["image_url"]
    .ne("")
    .astype("int8")
)


# ==========================================================
# 14. HANDLE COURSE URL
# ==========================================================

data["url"] = (
    data["url"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data["has_course_url"] = (
    data["url"]
    .ne("")
    .astype("int8")
)


# ==========================================================
# 15. HANDLE SOURCE FILE
# ==========================================================

data["source_file"] = (
    data["source_file"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)


# ==========================================================
# 16. RESET INDEX
# ==========================================================

data = data.reset_index(drop=True)

In [8]:
data.isnull().sum()

course_id                        0
course_name                      0
description                      0
skills                           0
subject                          0
level                            0
organization                     0
provider                         0
rating                           0
reviews_count                    0
students_enrolled                0
lectures_count                   0
duration                         0
instructor                       0
price                            0
language                         0
image_url                        0
url                              0
certificate_type                 0
course_type                      0
source_file                      0
course_name_was_missing          0
description_was_missing          0
skills_was_missing               0
subject_was_missing              0
level_was_missing                0
organization_was_missing         0
rating_was_missing               0
reviews_count_was_mi

In [9]:
data.columns

Index(['course_id', 'course_name', 'description', 'skills', 'subject', 'level',
       'organization', 'provider', 'rating', 'reviews_count',
       'students_enrolled', 'lectures_count', 'duration', 'instructor',
       'price', 'language', 'image_url', 'url', 'certificate_type',
       'course_type', 'source_file', 'course_name_was_missing',
       'description_was_missing', 'skills_was_missing', 'subject_was_missing',
       'level_was_missing', 'organization_was_missing', 'rating_was_missing',
       'reviews_count_was_missing', 'students_enrolled_was_missing',
       'lectures_count_was_missing', 'duration_was_missing',
       'instructor_was_missing', 'price_was_missing', 'language_was_missing',
       'image_url_was_missing', 'url_was_missing',
       'certificate_type_was_missing', 'course_type_was_missing', 'nlp_level',
       'nlp_language', 'nlp_certificate_type', 'nlp_course_type',
       'reviews_for_ranking', 'students_for_ranking', 'is_free', 'has_image',
       'has_cou

In [10]:
# Create an NLP cleaning function
def clean_nlp_text(value):
    if pd.isna(value):
        return ""

    text = html.unescape(str(value))
    text = text.casefold()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs inside text
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Protect important technical terms
    replacements = {
        "c++": "cplusplus",
        "c#": "csharp",
        ".net": "dotnet",
        "asp.net": "aspnet",
        "node.js": "nodejs",
        "react.js": "reactjs",
        "next.js": "nextjs",
        "vue.js": "vuejs",
        "scikit-learn": "scikitlearn",
        "tensorflow.js": "tensorflowjs",
        "ui/ux": "ui ux",
        "ai/ml": "artificial intelligence machine learning",
        "machine-learning": "machine learning",
        "deep-learning": "deep learning"
    }

    for old_value, new_value in replacements.items():
        text = text.replace(old_value, new_value)

    # Change underscores into spaces
    text = text.replace("_", " ")

    # Remove punctuation while preserving letters and numbers
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
print(
    clean_nlp_text(
        "Learn C++, C#, Node.js and Scikit-learn!"
    )
)

learn cplusplus csharp nodejs and scikitlearn


In [12]:
model_data = data.copy()

In [13]:
model_data = model_data.reset_index(drop=True)

In [14]:
# Clean the important NLP columns
text_columns = [
    "course_name",
    "description",
    "skills",
    "subject",
    "organization",
    "provider"
]

for column in text_columns:
    model_data[f"clean_{column}"] = (
        model_data[column].apply(clean_nlp_text)
    )

In [15]:
#Clean the NLP-safe categorical columns:
model_data["clean_level"] = (
    model_data["nlp_level"]
    .apply(clean_nlp_text)
)

model_data["clean_course_type"] = (
    model_data["nlp_course_type"]
    .apply(clean_nlp_text)
)

model_data["clean_certificate_type"] = (
    model_data["nlp_certificate_type"]
    .apply(clean_nlp_text)
)

In [16]:
# The tags column
model_data["tags"] = (
    # Course title: very important
    (model_data["clean_course_name"] + " ") * 4

    # Skills: very important
    + (model_data["clean_skills"] + " ") * 4

    # Subject: important
    + (model_data["clean_subject"] + " ") * 3

    # Description: moderately important
    + (model_data["clean_description"] + " ") * 2

    # Other course information
    + model_data["clean_level"] + " "
    + model_data["clean_course_type"] + " "
    + model_data["clean_certificate_type"] + " "
    + model_data["clean_organization"] + " "
    + model_data["clean_provider"]
)

In [17]:
# Remove repeated spaces:
model_data["tags"] = (
    model_data["tags"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [18]:
model_data[
    [
        "course_name",
        "skills",
        "subject",
        "tags"
    ]
].head()

,course_name,skills,subject,tags
0,Vector Calculus for Engineers,"Mathematical Modeling, Physics, Engineering, S...",Math and Logic,vector calculus for engineers vector calculus ...
1,Fibonacci Numbers and the Golden Ratio,"Recreational Mathematics, Discrete Mathematics...",Math and Logic,fibonacci numbers and the golden ratio fibonac...
2,Matrix Algebra for Engineers,"Algebra, Advanced Mathematics, Statistical Mod...",Math and Logic,matrix algebra for engineers matrix algebra fo...
3,Game Theory,,Economics,game theory game theory game theory game theor...
4,SQL for Data Science,"Data Science, Query Languages, Data Integratio...",Data Analysis,sql for data science sql for data science sql ...


In [19]:
row_number = 0

print("COURSE:")
print(model_data.loc[row_number, "course_name"])

print("\nTAGS:")
print(model_data.loc[row_number, "tags"])

COURSE:
Vector Calculus for Engineers

TAGS:
vector calculus for engineers vector calculus for engineers vector calculus for engineers vector calculus for engineers mathematical modeling physics engineering statistics advanced mathematics applied mathematics general mathematics mathematics and mathematical modeling mathematical theory analysis statistical modeling engineering analysis engineering calculations calculus integral calculus mathematical modeling physics engineering statistics advanced mathematics applied mathematics general mathematics mathematics and mathematical modeling mathematical theory analysis statistical modeling engineering analysis engineering calculations calculus integral calculus mathematical modeling physics engineering statistics advanced mathematics applied mathematics general mathematics mathematics and mathematical modeling mathematical theory analysis statistical modeling engineering analysis engineering calculations calculus integral calculus mathematic

In [20]:
# Remove unusable NLP records
empty_tags = (
    model_data["tags"].str.len() == 0
).sum()

print("Courses with empty tags:", empty_tags)

Courses with empty tags: 0


In [21]:
model_data = model_data[
    model_data["tags"].str.len() > 0
].reset_index(drop=True)

print("Model dataset shape:", model_data.shape)

Model dataset shape: (24646, 58)


In [22]:
# TF-IDF model
tfidf_vectorizer = TfidfVectorizer(
    # Capture words and two-word phrases
    ngram_range=(1, 2),

    # Ignore words that appear in only one course
    min_df=2,

    # Ignore terms appearing in over 95% of courses
    max_df=0.95,

    # Limit model size
    max_features=50_000,

    # Remove common English words
    stop_words="english",

    # Reduce the excessive effect of repeated words
    sublinear_tf=True,

    # Normalize vectors
    norm="l2",

    # Reduce memory use
    dtype=np.float32
)

In [23]:
tfidf_matrix = tfidf_vectorizer.fit_transform(
    model_data["tags"]
)

In [24]:
print("Number of courses:", tfidf_matrix.shape[0])
print("Number of NLP features:", tfidf_matrix.shape[1])
print("Matrix type:", type(tfidf_matrix))
print(
    "Vocabulary size:",
    len(tfidf_vectorizer.vocabulary_)
)

Number of courses: 24646
Number of NLP features: 50000
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Vocabulary size: 50000


In [25]:
def recommend_courses(
    query,
    top_n=10,
    level=None,
    provider=None,
    course_type=None,
    language=None,
    minimum_rating=None,
    free_only=False
):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Please enter a valid query.")

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    cleaned_query = clean_nlp_text(query)

    if not cleaned_query:
        raise ValueError(
            "The query does not contain usable words."
        )

    # Convert query into a TF-IDF vector
    query_vector = tfidf_vectorizer.transform(
        [cleaned_query]
    )

    # Compare query against every course
    similarity_scores = linear_kernel(
        query_vector,
        tfidf_matrix
    ).flatten()

    results = model_data.copy()

    results["similarity_score"] = similarity_scores

    # Create ranking columns when unavailable
    if "popularity_score" not in results.columns:
        results["popularity_score"] = 0.0

    if "data_quality_score" not in results.columns:
        results["data_quality_score"] = 0.0

    # Optional filters
    if level:
        results = results[
            results["level"]
            .fillna("")
            .astype(str)
            .str.casefold()
            .str.contains(
                str(level).casefold(),
                regex=False
            )
        ]

    if provider:
        results = results[
            results["provider"]
            .fillna("")
            .astype(str)
            .str.casefold()
            .eq(str(provider).casefold())
        ]

    if course_type:
        results = results[
            results["course_type"]
            .fillna("")
            .astype(str)
            .str.casefold()
            .str.contains(
                str(course_type).casefold(),
                regex=False
            )
        ]

    if language:
        results = results[
            results["language"]
            .fillna("")
            .astype(str)
            .str.casefold()
            .str.contains(
                str(language).casefold(),
                regex=False
            )
        ]

    if minimum_rating is not None:
        results = results[
            pd.to_numeric(
                results["rating"],
                errors="coerce"
            ) >= float(minimum_rating)
        ]

    if free_only:
        results = results[
            results["is_free"] == 1
        ]

    # Remove completely unrelated results
    results = results[
        results["similarity_score"] > 0
    ].copy()

    # Calculate final combined score
    results["final_score"] = (
        0.85 * results["similarity_score"]
        + 0.10 * results["popularity_score"]
        + 0.05 * results["data_quality_score"]
    )

    # Rank courses
    results = results.sort_values(
        by=["final_score", "similarity_score"],
        ascending=False
    ).head(top_n)

    output_columns = [
        "course_id",
        "course_name",
        "description",
        "skills",
        "subject",
        "level",
        "organization",
        "provider",
        "rating",
        "reviews_count",
        "students_enrolled",
        "duration",
        "instructor",
        "price",
        "language",
        "image_url",
        "url",
        "certificate_type",
        "course_type",
        "is_free",
        "similarity_score",
        "popularity_score",
        "data_quality_score",
        "final_score"
    ]

    available_columns = [
        column
        for column in output_columns
        if column in results.columns
    ]

    return results[
        available_columns
    ].reset_index(drop=True)

In [26]:
recommendations = recommend_courses(
    query="beginner Python data analysis using pandas",
    top_n=10
)

recommendations[
    [
        "course_name",
        "skills",
        "subject",
        "level",
        "provider",
        "rating",
        "similarity_score"
    ]
]

,course_name,skills,subject,level,provider,rating,similarity_score
0,Pandas with Python for Data Science,,,All Levels,Udemy,4.7,0.470392
1,Data Analysis with Pandas and Python,,,All Levels,Udemy,4.6,0.461845
2,Statistics for Data Analysis Using R,,,All Levels,Udemy,4.7,0.431722
3,Python for Data Science and Data Analysis,,,All Levels,Udemy,4.2,0.406338
4,Crash course: Data analytics in Python using P...,,,Beginner,Udemy,4.5,0.405482
5,Complete Data Science Training with Python for...,,,All Levels,Udemy,4.4,0.370480
6,Beginner's Guide to Python Data Analysis & Vis...,,,Beginner,Udemy,3.9,0.358573
7,Learning Python for Data Analysis and Visualiz...,,,All Levels,Udemy,4.3,0.352913
8,Python for Data Analysis: step-by-step with pr...,,,All Levels,Udemy,4.6,0.351011
9,Python for Data Analysis & Visualization 2022,,,All Levels,Udemy,4.5,0.328904


In [27]:
recommendations = recommend_courses(
    query="machine learning with Python",
    top_n=10,
    level="Beginner",
    provider="Coursera",
    minimum_rating=4.0
)

recommendations[
    [
        "course_name",
        "level",
        "provider",
        "rating",
        "similarity_score"
    ]
]

,course_name,level,provider,rating,similarity_score
0,Deep Neural Networks with PyTorch,Beginner,Coursera,4.4,0.317981
1,Machine Learning,Beginner,Coursera,4.9,0.298631
2,IBM Applied AI,Beginner,Coursera,4.6,0.288129
3,Applied Data Science,Beginner,Coursera,4.6,0.269255
4,IBM Data Science,Beginner,Coursera,4.6,0.259804
5,Local LLMs with llamafile,Beginner,Coursera,4.7,0.253249
6,Supervised Machine Learning: Regression and Cl...,Beginner,Coursera,4.9,0.220720
7,MLOps | Machine Learning Operations,Beginner,Coursera,4.0,0.211380
8,Auto Machine Learning (AutoML) Using AutoGluon,Beginner,Coursera,4.7,0.207550
9,Compare Models with Experiments in Azure ML St...,Beginner,Coursera,4.7,0.194567


In [28]:
recommendations = recommend_courses(
    query="web development with JavaScript",
    top_n=10,
    free_only=True
)

In [29]:
# similar-course function
def recommend_similar_courses(
    course_id,
    top_n=10
):
    """
    Recommend courses similar to a selected course.
    """

    matching_indices = model_data.index[
        model_data["course_id"]
        .astype(str)
        .eq(str(course_id))
    ].tolist()

    if not matching_indices:
        raise ValueError(
            f"No course found with course_id={course_id}"
        )

    selected_index = matching_indices[0]

    similarity_scores = linear_kernel(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = similarity_scores.argsort()[::-1]

    # Remove the selected course itself
    ranked_indices = [
        index
        for index in ranked_indices
        if index != selected_index
    ][:top_n]

    results = model_data.iloc[
        ranked_indices
    ].copy()

    results["similarity_score"] = (
        similarity_scores[ranked_indices]
    )

    output_columns = [
        "course_id",
        "course_name",
        "description",
        "skills",
        "subject",
        "level",
        "organization",
        "provider",
        "rating",
        "duration",
        "price",
        "image_url",
        "url",
        "similarity_score"
    ]

    available_columns = [
        column
        for column in output_columns
        if column in results.columns
    ]

    return results[
        available_columns
    ].reset_index(drop=True)

In [30]:
selected_course_id = model_data.iloc[0]["course_id"]

similar_courses = recommend_similar_courses(
    course_id=selected_course_id,
    top_n=10
)

print(
    "Selected course:",
    model_data.iloc[0]["course_name"]
)

similar_courses[
    [
        "course_name",
        "skills",
        "subject",
        "provider",
        "similarity_score"
    ]
]

Selected course: Vector Calculus for Engineers


,course_name,skills,subject,provider,similarity_score
0,Matrix Algebra for Engineers,"Algebra, Advanced Mathematics, Statistical Mod...",Math and Logic,Coursera,0.432996
1,Mathematics for Engineers: The Capstone Course,"Numerical Analysis, Finite Element Methods, Ap...",,Coursera,0.318631
2,Integral Calculus and Numerical Analysis for D...,"General Mathematics, Linear Algebra, Applied M...",,Coursera,0.308894
3,Introduction to Mathematical Thinking,"Logical Reasoning, Analytical Skills, Problem ...",Math and Logic,Coursera,0.304246
4,Physics of Oscillators and Waves,"Mathematical Modeling, Advanced Mathematics, P...",,Coursera,0.268002
5,Mathematics for Engineers Specialization,,Math and Logic,Coursera,0.259746
6,Single Variable Calculus,"Integral Calculus, Applied Mathematics, Mathem...",,Coursera,0.258234
7,Calculus through Data & Modelling: Vector Calc...,,,Coursera,0.254669
8,Physics 102 - AC Circuits and Maxwell's Equations,"Mathematics and Mathematical Modeling, Physics...",,Coursera,0.251865
9,Quantitative Foundations for International Bus...,"Business Mathematics, Mathematical Theory & An...",,Coursera,0.249953


In [31]:
test_queries = [
    "Python programming for beginners",
    "machine learning and artificial intelligence",
    "data analysis using Excel",
    "cybersecurity and ethical hacking",
    "Flutter mobile application development",
    "digital marketing and social media",
    "financial accounting and investment",
    "project management certification",
    "graphic design using Photoshop",
    "mathematics for engineers"
]

for query in test_queries:
    results = recommend_courses(
        query=query,
        top_n=5
    )

    print("\nQUERY:", query)

    for _, course in results.iterrows():
        print(
            f"- {course['course_name']} "
            f"({course['similarity_score']:.3f})"
        )


QUERY: Python programming for beginners
- Python Programming for Beginners (0.819)
- Python Programming - For Every Beginners (0.816)
- Perl Programming for Beginners (0.786)
- Hello Python - Python Programming for Beginners (0.728)
- Python Programming for Beginners and Kids - Anyone Can Code (0.688)

QUERY: machine learning and artificial intelligence
- Machine Learning and Artificial Intelligence Using Swift (0.681)
- Artificial Intelligence & Machine Learning for Business (0.629)
- Artificial Intelligence Introduction for Non-Technicals (0.529)
- Tensorflow 2.0: Deep Learning and Artificial Intelligence (0.497)
- Artificial Intelligence Masterclass (0.468)

QUERY: data analysis using Excel
- Statistics for Data Analysis Using Excel 2016 (0.771)
- Statistics for Data Analysis Using R (0.536)
- Financial Ratios Using Excel (0.362)
- Technical Analysis Using Elliott Wave Theory (0.296)
- Excel Basics for Data Analysis (0.284)

QUERY: cybersecurity and ethical hacking
- Learn Python &

In [32]:
models_directory = Path("models")
models_directory.mkdir(
    parents=True,
    exist_ok=True #Save the NLP model
)

In [33]:
joblib.dump(
    tfidf_vectorizer,
    models_directory / "tfidf_vectorizer.joblib" # Save the vectorizer:
)

['models\\tfidf_vectorizer.joblib']

In [34]:
# Save the TF-IDF matrix:

joblib.dump(
    tfidf_matrix,
    models_directory / "tfidf_matrix.joblib"
)

['models\\tfidf_matrix.joblib']

In [35]:
temporary_columns = [
    column
    for column in model_data.columns
    if column.startswith("clean_")
]

In [36]:
courses_to_save = model_data.drop(
    columns=temporary_columns + ["tags"],
    errors="ignore" # Final course data 
)

In [37]:
joblib.dump(
    courses_to_save,
    models_directory / "courses.joblib"
)

['models\\courses.joblib']

In [38]:
recommendations = recommend_courses(
    query="beginner Python data analysis using pandas",
    top_n=10
)

recommendations[
    [
        "course_name",
        "skills",
        "subject",
        "level",
        "provider",
        "rating",
        "similarity_score",
        "final_score"
    ]
]

,course_name,skills,subject,level,provider,rating,similarity_score,final_score
0,Pandas with Python for Data Science,,,All Levels,Udemy,4.7,0.470392,0.399833
1,Data Analysis with Pandas and Python,,,All Levels,Udemy,4.6,0.461845,0.392568
2,Statistics for Data Analysis Using R,,,All Levels,Udemy,4.7,0.431722,0.366964
3,Python for Data Science and Data Analysis,,,All Levels,Udemy,4.2,0.406338,0.345387
4,Crash course: Data analytics in Python using P...,,,Beginner,Udemy,4.5,0.405482,0.344660
5,Complete Data Science Training with Python for...,,,All Levels,Udemy,4.4,0.370480,0.314908
6,Beginner's Guide to Python Data Analysis & Vis...,,,Beginner,Udemy,3.9,0.358573,0.304787
7,Learning Python for Data Analysis and Visualiz...,,,All Levels,Udemy,4.3,0.352913,0.299976
8,Python for Data Analysis: step-by-step with pr...,,,All Levels,Udemy,4.6,0.351011,0.298359
9,Python for Data Analysis & Visualization 2022,,,All Levels,Udemy,4.5,0.328904,0.279568


In [39]:
recommendations = recommend_courses(
    query="cybersecurity and ethical hacking",
    top_n=5
)

recommendations[
    [
        "course_name",
        "skills",
        "provider",
        "similarity_score",
        "final_score"
    ]
]

,course_name,skills,provider,similarity_score,final_score
0,Learn Python & Ethical Hacking From Scratch,,Udemy,0.638531,0.542751
1,Build Undetectable Malware Using C Language: E...,,Udemy,0.623911,0.530324
2,The Complete Ethical Hacking Course 2.0: Pytho...,,Udemy,0.508362,0.432108
3,Hacking the Bootcamp,,Udemy,0.388913,0.330576
4,Ethical Hacking Fundamentals,"Penetration Testing, Ethical Hacking, Footprin...",Coursera,0.326944,0.277902


In [40]:
selected_course_id = model_data.iloc[0]["course_id"]

print(
    "Selected course:",
    model_data.iloc[0]["course_name"]
)

similar_courses = recommend_similar_courses(
    course_id=selected_course_id,
    top_n=10
)

similar_courses[
    [
        "course_name",
        "skills",
        "provider",
        "similarity_score"
    ]
]

Selected course: Vector Calculus for Engineers


,course_name,skills,provider,similarity_score
0,Matrix Algebra for Engineers,"Algebra, Advanced Mathematics, Statistical Mod...",Coursera,0.432996
1,Mathematics for Engineers: The Capstone Course,"Numerical Analysis, Finite Element Methods, Ap...",Coursera,0.318631
2,Integral Calculus and Numerical Analysis for D...,"General Mathematics, Linear Algebra, Applied M...",Coursera,0.308894
3,Introduction to Mathematical Thinking,"Logical Reasoning, Analytical Skills, Problem ...",Coursera,0.304246
4,Physics of Oscillators and Waves,"Mathematical Modeling, Advanced Mathematics, P...",Coursera,0.268002
5,Mathematics for Engineers Specialization,,Coursera,0.259746
6,Single Variable Calculus,"Integral Calculus, Applied Mathematics, Mathem...",Coursera,0.258234
7,Calculus through Data & Modelling: Vector Calc...,,Coursera,0.254669
8,Physics 102 - AC Circuits and Maxwell's Equations,"Mathematics and Mathematical Modeling, Physics...",Coursera,0.251865
9,Quantitative Foundations for International Bus...,"Business Mathematics, Mathematical Theory & An...",Coursera,0.249953


In [41]:
from pathlib import Path
import json
import joblib

In [42]:
MODELS_DIR = Path("models")

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", MODELS_DIR.resolve())

Model directory: D:\Course_recomendation_Deep_Learning_Extended\models


In [43]:
joblib.dump(
    tfidf_vectorizer,
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

print("TF-IDF vectorizer saved.")

TF-IDF vectorizer saved.


In [44]:
joblib.dump(
    tfidf_matrix,
    MODELS_DIR / "tfidf_matrix.joblib"
)

print("TF-IDF matrix saved.")

TF-IDF matrix saved.


In [45]:
print(
    "popularity_score exists:",
    "popularity_score" in courses_to_save.columns
)

print(
    "data_quality_score exists:",
    "data_quality_score" in courses_to_save.columns
)

popularity_score exists: False
data_quality_score exists: False


In [46]:
if "popularity_score" not in model_data.columns:
    model_data["popularity_score"] = 0.0

if "data_quality_score" not in model_data.columns:
    model_data["data_quality_score"] = 0.0

In [47]:
temporary_columns = [
    column
    for column in model_data.columns
    if column.startswith("clean_")
]

print("Temporary columns that will be removed:")
print(temporary_columns)

Temporary columns that will be removed:
['clean_course_name', 'clean_description', 'clean_skills', 'clean_subject', 'clean_organization', 'clean_provider', 'clean_level', 'clean_course_type', 'clean_certificate_type']


In [48]:
courses_to_save = model_data.drop(
    columns=temporary_columns + ["tags"],
    errors="ignore"
).copy()

print("Original model data shape:", model_data.shape)
print("Saved course data shape:", courses_to_save.shape)

Original model data shape: (24646, 60)
Saved course data shape: (24646, 50)


In [49]:
joblib.dump(
    courses_to_save,
    MODELS_DIR / "courses.joblib"
)

print("Course data saved.")

Course data saved.


In [50]:
metadata = {
    "model_name": "EduCompass TF-IDF Course Recommender",
    "model_type": "Content-based NLP recommendation system",
    "number_of_courses": int(tfidf_matrix.shape[0]),
    "number_of_features": int(tfidf_matrix.shape[1]),
    "vectorizer": "TfidfVectorizer",
    "ngram_range": [1, 2],
    "max_features": 50000,
    "recommendation_methods": [
        "Natural-language query recommendation",
        "Similar-course recommendation"
    ]
}

with open(
    MODELS_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=4
    )

print("Model metadata saved.")

Model metadata saved.


In [51]:

print("Model saved successfully.\n")
print("Saved files:")

for file_path in sorted(MODELS_DIR.iterdir()):
    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"- {file_path.name}: "
        f"{file_size_mb:.2f} MB"
    )

Model saved successfully.

Saved files:
- courses.joblib: 49.09 MB
- model_metadata.json: 0.00 MB
- PUT_MODEL_FILES_HERE.txt: 0.00 MB
- tfidf_matrix.joblib: 24.06 MB
- tfidf_vectorizer.joblib: 1.83 MB


In [52]:
loaded_vectorizer = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

loaded_matrix = joblib.load(
    MODELS_DIR / "tfidf_matrix.joblib"
)

loaded_courses = joblib.load(
    MODELS_DIR / "courses.joblib"
)

print("Loaded courses shape:", loaded_courses.shape)
print("Loaded matrix shape:", loaded_matrix.shape)
print(
    "Loaded vocabulary size:",
    len(loaded_vectorizer.vocabulary_)
)

assert loaded_courses.shape[0] == loaded_matrix.shape[0]

print("Verification passed.")

Loaded courses shape: (24646, 50)
Loaded matrix shape: (24646, 50000)
Loaded vocabulary size: 50000
Verification passed.


In [53]:
assert loaded_courses.shape[0] == loaded_matrix.shape[0]

print("Verification passed: course rows match TF-IDF rows.")

Verification passed: course rows match TF-IDF rows.


## Why this model uses ranking metrics

This course recommender is content-based and returns a ranked list of similar courses, so classification accuracy is not a meaningful evaluation metric.

The cell below reports offline ranking metrics instead:
- Precision@10
- Recall@10
- MRR@10
- NDCG@10

In [7]:
baseline_config = {
    "model_name": "word_tfidf_baseline",
    "ngram_range": [1, 2],
    "max_features": 50_000,
    "similarity_weight": 0.85,
    "popularity_weight": 0.10,
    "quality_weight": 0.05,
}

ai_assisted_relevance_judgments = {
    "beginner Python programming": [
        22,
        41,
        43,
        22785,
    ],
    "machine learning with Python": [
        48,
        5326,
        5344,
        14761,
    ],
    "digital marketing for beginners": [
        3443,
        6768,
        8573,
    ],
    "project management certification": [
        138,
        706,
        13617,
        15245,
    ],
    "cybersecurity and ethical hacking": [
        322,
        1584,
        7023,
        7026,
    ],
    "data visualization": [
        6185,
        6228,
        6231,
        6236,
        6255,
    ],
}

print("Baseline configuration frozen:")
print(baseline_config)
print("\nAI-assisted relevance queries prepared:", len(ai_assisted_relevance_judgments))

Baseline configuration frozen:
{'model_name': 'word_tfidf_baseline', 'ngram_range': [1, 2], 'max_features': 50000, 'similarity_weight': 0.85, 'popularity_weight': 0.1, 'quality_weight': 0.05}

AI-assisted relevance queries prepared: 6


In [8]:
from __future__ import annotations

from pathlib import Path
import html
import json
import math
import re
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import linear_kernel

BASE_DIR = Path(r"D:\Course_recomendation_Deep_Learning_Extended")
MODELS_DIR = BASE_DIR / "models"
EVAL_DIR = MODELS_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

baseline_config = {
    "model_name": "word_tfidf_baseline",
    "ngram_range": [1, 2],
    "max_features": 50_000,
    "similarity_weight": 0.85,
    "popularity_weight": 0.10,
    "quality_weight": 0.05,
}

baseline_config_path = EVAL_DIR / "baseline_config.json"
with open(baseline_config_path, "w", encoding="utf-8") as file:
    json.dump(baseline_config, file, indent=4)

baseline_vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
baseline_matrix = joblib.load(MODELS_DIR / "tfidf_matrix.joblib")
baseline_courses = joblib.load(MODELS_DIR / "courses.joblib").copy()

baseline_courses["course_id"] = baseline_courses["course_id"].astype(str)

assert baseline_courses["course_id"].is_unique, "course_id must be unique in the saved course table."
assert baseline_courses.shape[0] == baseline_matrix.shape[0], "courses.joblib rows must match TF-IDF rows."

for required_column in ["course_name", "description", "skills", "subject", "level", "course_type", "certificate_type", "provider", "organization", "url"]:
    if required_column not in baseline_courses.columns:
        baseline_courses[required_column] = ""

if "popularity_score" not in baseline_courses.columns:
    baseline_courses["popularity_score"] = 0.0

if "data_quality_score" not in baseline_courses.columns:
    baseline_courses["data_quality_score"] = 0.0

baseline_courses["popularity_score"] = pd.to_numeric(baseline_courses["popularity_score"], errors="coerce").fillna(0.0)
baseline_courses["data_quality_score"] = pd.to_numeric(baseline_courses["data_quality_score"], errors="coerce").fillna(0.0)

TEXT_FIELDS = [
    "course_name",
    "description",
    "skills",
    "subject",
    "organization",
    "provider",
    "level",
    "course_type",
    "certificate_type",
]

QUERY_SPECS = [
    "beginner Python programming",
    "advanced Python automation",
    "Python data analysis with pandas",
    "machine learning with Python",
    "deep learning with TensorFlow",
    "data science with visualization",
    "exploratory data analysis for beginners",
    "web development with JavaScript",
    "React front-end development",
    "mobile app development with Flutter",
    "Android app development with Kotlin",
    "cybersecurity and ethical hacking",
    "network security certification",
    "cloud computing with AWS",
    "Google Cloud architecture certification",
    "SQL database design for beginners",
    "PostgreSQL data engineering",
    "digital marketing for beginners",
    "social media marketing analytics",
    "business analytics for decision making",
    "business strategy and management",
    "finance for managers",
    "financial accounting and investment analysis",
    "project management certification",
    "agile project management for beginners",
    "graphic design with Adobe Photoshop",
    "UI UX design portfolio",
    "calculus and linear algebra for engineers",
    "statistics and probability for data science",
    "learn Spanish for beginners",
    "English grammar and vocabulary",
    "programming for complete beginners",
    "intro to data analysis with Excel",
    "advanced machine learning model deployment",
    "advanced cloud security engineering",
    "AWS certification preparation",
    "CompTIA Security+ practice",
]

LEVEL_HINTS = {
    "beginner": "beginner",
    "intro": "beginner",
    "advanced": "advanced",
    "intermediate": "intermediate",
}

CERT_HINTS = {
    "certification",
    "certificate",
    "certified",
    "practice exam",
    "compTIA",
    "aws",
    "security+",
}


def clean_nlp_text(value) -> str:
    if pd.isna(value):
        return ""

    text = html.unescape(str(value))
    text = text.casefold()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    replacements = {
        "c++": "cplusplus",
        "c#": "csharp",
        ".net": "dotnet",
        "asp.net": "aspnet",
        "node.js": "nodejs",
        "react.js": "reactjs",
        "next.js": "nextjs",
        "vue.js": "vuejs",
        "scikit-learn": "scikitlearn",
        "tensorflow.js": "tensorflowjs",
        "ui/ux": "ui ux",
        "ai/ml": "artificial intelligence machine learning",
        "machine-learning": "machine learning",
        "deep-learning": "deep learning",
    }

    for original_text, replacement_text in replacements.items():
        text = text.replace(original_text, replacement_text)

    text = text.replace("_", " ")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_query(query: str) -> list[str]:
    cleaned_query = clean_nlp_text(query)
    return [token for token in cleaned_query.split() if token]


def combined_text_frame(dataframe: pd.DataFrame) -> pd.Series:
    pieces = []
    for field_name in TEXT_FIELDS:
        pieces.append(dataframe[field_name].fillna("").astype(str))
    return (
        pieces[0]
        + " " + pieces[1]
        + " " + pieces[2]
        + " " + pieces[3]
        + " " + pieces[4]
        + " " + pieces[5]
        + " " + pieces[6]
        + " " + pieces[7]
        + " " + pieces[8]
    ).str.replace(r"\s+", " ", regex=True).str.strip()


baseline_courses["_combined_text"] = combined_text_frame(baseline_courses)
baseline_courses["_combined_text_clean"] = baseline_courses["_combined_text"].apply(clean_nlp_text)


def recommend_baseline(query: str, top_n: int = 10) -> pd.DataFrame:
    cleaned_query = clean_nlp_text(query)
    if not cleaned_query:
        raise ValueError("The query does not contain usable words.")

    query_vector = baseline_vectorizer.transform([cleaned_query])
    similarity_scores = linear_kernel(query_vector, baseline_matrix).ravel()

    results = baseline_courses.copy()
    results["similarity_score"] = similarity_scores
    results["final_score"] = (
        baseline_config["similarity_weight"] * results["similarity_score"]
        + baseline_config["popularity_weight"] * results["popularity_score"]
        + baseline_config["quality_weight"] * results["data_quality_score"]
    )

    results = results[results["similarity_score"] > 0].copy()
    results = results.sort_values(
        by=["final_score", "similarity_score"],
        ascending=False,
    ).head(top_n)

    return results.reset_index(drop=True)


def lexical_search(query: str, top_n: int = 10) -> pd.DataFrame:
    tokens = tokenize_query(query)
    if not tokens:
        return baseline_courses.head(0).copy()

    working = baseline_courses.copy()
    score = pd.Series(0, index=working.index, dtype="int64")
    for token in tokens:
        token_pattern = re.escape(token)
        score += working["_combined_text_clean"].str.contains(token_pattern, regex=True).astype(int)

    working["lexical_score"] = score
    return working.sort_values(
        by=["lexical_score", "popularity_score"],
        ascending=False,
    ).head(top_n).reset_index(drop=True)


def subject_skill_search(query: str, top_n: int = 10) -> pd.DataFrame:
    tokens = tokenize_query(query)
    if not tokens:
        return baseline_courses.head(0).copy()

    working = baseline_courses.copy()
    score = pd.Series(0, index=working.index, dtype="int64")
    for token in tokens:
        token_pattern = re.escape(token)
        score += working["subject"].fillna("").astype(str).str.casefold().str.contains(token_pattern, regex=True).astype(int) * 2
        score += working["skills"].fillna("").astype(str).str.casefold().str.contains(token_pattern, regex=True).astype(int) * 2
        score += working["course_name"].fillna("").astype(str).str.casefold().str.contains(token_pattern, regex=True).astype(int) * 3

    working["subject_skill_score"] = score
    return working.sort_values(
        by=["subject_skill_score", "popularity_score"],
        ascending=False,
    ).head(top_n).reset_index(drop=True)


def hard_negative_search(query: str, exclude_ids: set[str], top_n: int = 10, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(abs(hash(query)) % (2**32) if seed is None else seed)
    candidates = baseline_courses.loc[~baseline_courses["course_id"].isin(exclude_ids)].copy()
    if candidates.empty:
        return candidates

    tokens = tokenize_query(query)
    if tokens:
        token_hits = pd.Series(0, index=candidates.index, dtype="int64")
        for token in tokens:
            token_pattern = re.escape(token)
            token_hits += candidates["_combined_text_clean"].str.contains(token_pattern, regex=True).astype(int)
        candidates = candidates.assign(_token_hits=token_hits)
    else:
        candidates = candidates.assign(_token_hits=0)

    # Prefer low lexical overlap but keep some same-level / same-provider items as hard negatives.
    candidates = candidates.sort_values(by=["_token_hits", "popularity_score"], ascending=[True, False])
    pool = candidates.head(max(top_n * 5, 25)).copy()
    if len(pool) <= top_n:
        return pool.head(top_n).reset_index(drop=True)

    sample_indices = rng.choice(pool.index.to_numpy(), size=top_n, replace=False)
    return pool.loc[sample_indices].reset_index(drop=True)


def build_candidate_pool(query: str, top_n: int = 20) -> pd.DataFrame:
    baseline_candidates = recommend_baseline(query, top_n=6)
    lexical_candidates = lexical_search(query, top_n=6)
    subject_skill_candidates = subject_skill_search(query, top_n=6)

    exclude_ids = set(baseline_candidates["course_id"].astype(str).tolist())
    exclude_ids.update(lexical_candidates["course_id"].astype(str).tolist())
    exclude_ids.update(subject_skill_candidates["course_id"].astype(str).tolist())

    negative_candidates = hard_negative_search(query, exclude_ids=exclude_ids, top_n=6, seed=42)

    candidate_frames = [
        baseline_candidates,
        lexical_candidates,
        subject_skill_candidates,
        negative_candidates,
    ]

    combined = pd.concat(candidate_frames, ignore_index=True, sort=False)
    combined["course_id"] = combined["course_id"].astype(str)
    combined = combined.drop_duplicates(subset=["course_id"], keep="first")
    combined = combined.head(top_n).copy()
    return combined.reset_index(drop=True)


def grade_candidate(query: str, row: pd.Series) -> tuple[int, str, float]:
    tokens = tokenize_query(query)
    query_text = clean_nlp_text(query)
    course_name = clean_nlp_text(row.get("course_name", ""))
    skills = clean_nlp_text(row.get("skills", ""))
    subject = clean_nlp_text(row.get("subject", ""))
    description = clean_nlp_text(row.get("description", ""))
    level = clean_nlp_text(row.get("level", ""))
    course_type = clean_nlp_text(row.get("course_type", ""))
    certificate_type = clean_nlp_text(row.get("certificate_type", ""))
    provider = clean_nlp_text(row.get("provider", ""))
    organization = clean_nlp_text(row.get("organization", ""))

    title_hits = sum(token in course_name for token in tokens)
    skill_hits = sum(token in skills for token in tokens)
    subject_hits = sum(token in subject for token in tokens)
    description_hits = sum(token in description for token in tokens)

    topical_score = title_hits * 3 + skill_hits * 2 + subject_hits * 2 + description_hits

    level_hint = None
    for hint_text, hint_level in LEVEL_HINTS.items():
        if hint_text in query_text:
            level_hint = hint_level
            break

    cert_hint = any(hint in query_text for hint in ["certification", "certificate", "certified", "practice exam", "security+", "aws"])
    provider_hint = None
    for candidate_provider in ["coursera", "udemy", "edx", "google", "ibm", "aws", "microsoft"]:
        if candidate_provider in query_text:
            provider_hint = candidate_provider
            break

    level_match = bool(level_hint and level_hint in level)
    cert_match = bool(cert_hint and ("cert" in certificate_type or "course" in course_type or "specialization" in course_type))
    provider_match = bool(provider_hint and provider_hint in provider)

    if topical_score >= 8 or (title_hits >= 2 and skill_hits >= 1):
        grade = 3
    elif topical_score >= 4 or (title_hits >= 1 and (skill_hits >= 1 or subject_hits >= 1)):
        grade = 2
    elif topical_score >= 1 or level_match or cert_match or provider_match:
        grade = 1
    else:
        grade = 0

    if level_match:
        grade = max(grade, 2)
    if cert_match and grade > 0:
        grade = max(grade, 2)
    if provider_match and grade > 0:
        grade = max(grade, 1)

    explanation_parts = []
    if title_hits:
        explanation_parts.append(f"title matches={title_hits}")
    if skill_hits:
        explanation_parts.append(f"skills matches={skill_hits}")
    if subject_hits:
        explanation_parts.append(f"subject matches={subject_hits}")
    if description_hits:
        explanation_parts.append(f"description matches={description_hits}")
    if level_match:
        explanation_parts.append(f"level match={row.get('level', '')}")
    if cert_match:
        explanation_parts.append(f"certification match={row.get('certificate_type', '') or row.get('course_type', '')}")
    if provider_match:
        explanation_parts.append(f"provider match={row.get('provider', '')}")

    explanation = "; ".join(explanation_parts) if explanation_parts else "No strong topical match found."
    confidence = float(min(0.95, 0.45 + 0.08 * topical_score + (0.08 if level_match else 0.0) + (0.08 if cert_match else 0.0)))
    return grade, explanation, confidence


validation_queries = QUERY_SPECS
validation_rows = []
for query in validation_queries:
    candidate_pool = build_candidate_pool(query, top_n=18)
    for _, candidate in candidate_pool.iterrows():
        grade, explanation, confidence = grade_candidate(query, candidate)
        validation_rows.append(
            {
                "query": query,
                "course_id": str(candidate["course_id"]),
                "course_name": candidate["course_name"],
                "relevance_grade": int(grade),
                "explanation": explanation,
                "confidence": round(confidence, 3),
                "judgment_source": "AI-assisted",
            }
        )

ai_assisted_relevance_judgments = pd.DataFrame(validation_rows)
ai_assisted_relevance_judgments = ai_assisted_relevance_judgments.drop_duplicates(
    subset=["query", "course_id"],
    keep="first",
).reset_index(drop=True)

review_table = ai_assisted_relevance_judgments.copy()
review_table["human_reviewed_grade"] = pd.NA
review_table["human_notes"] = ""
review_table["review_status"] = "needs_human_review"

judgments_path = EVAL_DIR / "ai_assisted_relevance_judgments.csv"
review_path = EVAL_DIR / "ai_assisted_relevance_review_table.csv"
ai_assisted_relevance_judgments.to_csv(judgments_path, index=False)
review_table.to_csv(review_path, index=False)

print(f"AI-assisted relevance judgments saved: {judgments_path}")
print(f"Review table saved: {review_path}")
print(f"Queries covered: {ai_assisted_relevance_judgments['query'].nunique()}")
print(f"Judgments rows: {len(ai_assisted_relevance_judgments)}")


def precision_at_k(relevance_grades: list[int], k: int) -> float:
    if k <= 0:
        raise ValueError("k must be positive.")
    top_k = relevance_grades[:k]
    return float(sum(grade > 0 for grade in top_k) / k)


def recall_at_k(relevance_grades: list[int], k: int, total_relevant: int) -> float:
    if k <= 0:
        raise ValueError("k must be positive.")
    if total_relevant <= 0:
        warnings.warn("Query has no relevant items; recall is undefined.")
        return 0.0
    top_k = relevance_grades[:k]
    return float(sum(grade > 0 for grade in top_k) / total_relevant)


def reciprocal_rank(relevance_grades: list[int]) -> float:
    for rank, grade in enumerate(relevance_grades, start=1):
        if grade > 0:
            return float(1.0 / rank)
    return 0.0


def mean_reciprocal_rank(per_query_rr: list[float]) -> float:
    return float(np.mean(per_query_rr)) if per_query_rr else 0.0


def dcg_at_k(relevance_grades: list[int], k: int) -> float:
    if k <= 0:
        raise ValueError("k must be positive.")
    graded = np.asarray(relevance_grades[:k], dtype=float)
    if graded.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, graded.size + 2))
    gains = (2.0 ** graded - 1.0) / discounts
    return float(np.sum(gains))


def ndcg_at_k(relevance_grades: list[int], score_values: list[float], k: int) -> float:
    if k <= 0:
        raise ValueError("k must be positive.")
    true = np.asarray(relevance_grades, dtype=float).reshape(1, -1)
    scores = np.asarray(score_values, dtype=float).reshape(1, -1)
    value = float(ndcg_score(true, scores, k=k))
    if not 0.0 <= value <= 1.0:
        raise AssertionError("NDCG must be between 0 and 1.")
    return value


def evaluate_query_recommendations(query: str, top_n: int = 10) -> pd.DataFrame:
    recommendations = recommend_baseline(query, top_n=top_n)
    recommended_ids = recommendations["course_id"].astype(str).tolist()
    scores = recommendations["final_score"].astype(float).tolist()

    query_judgments = ai_assisted_relevance_judgments[
        ai_assisted_relevance_judgments["query"].eq(query)
    ].copy()
    query_judgments["course_id"] = query_judgments["course_id"].astype(str)

    if query_judgments.empty:
        warnings.warn(f"No AI-assisted relevance judgments found for query: {query}")
        return pd.DataFrame()

    judged_lookup = query_judgments.set_index("course_id")["relevance_grade"].to_dict()
    relevance_grades = [int(judged_lookup.get(course_id, 0)) for course_id in recommended_ids]
    total_relevant = int((query_judgments["relevance_grade"] > 0).sum())

    return pd.DataFrame(
        [
            {
                "query": query,
                "precision_at_5": precision_at_k(relevance_grades, 5),
                "recall_at_10": recall_at_k(relevance_grades, 10, total_relevant),
                "reciprocal_rank": reciprocal_rank(relevance_grades),
                "ndcg_at_10": ndcg_at_k(relevance_grades, scores, 10),
                "number_of_relevant_courses": total_relevant,
                "number_of_retrieved_courses": len(recommended_ids),
            }
        ]
    )


per_query_results = []
for query in validation_queries:
    per_query_results.append(evaluate_query_recommendations(query, top_n=10))

baseline_results = pd.concat(per_query_results, ignore_index=True)
baseline_results["model_version"] = baseline_config["model_name"]

baseline_results = baseline_results[
    [
        "model_version",
        "query",
        "precision_at_5",
        "recall_at_10",
        "reciprocal_rank",
        "ndcg_at_10",
        "number_of_relevant_courses",
        "number_of_retrieved_courses",
    ]
].copy()

baseline_results_path = EVAL_DIR / "baseline_results.csv"
baseline_results.to_csv(baseline_results_path, index=False)

baseline_summary = {
    "queries_evaluated": int(baseline_results["query"].nunique()),
    "precision_at_5": float(baseline_results["precision_at_5"].mean()),
    "recall_at_10": float(baseline_results["recall_at_10"].mean()),
    "mrr": float(baseline_results["reciprocal_rank"].mean()),
    "ndcg_at_10": float(baseline_results["ndcg_at_10"].mean()),
}

print("Baseline results saved:", baseline_results_path)
print("Baseline summary:")
print(baseline_summary)


AI-assisted relevance judgments saved: D:\Course_recomendation_Deep_Learning_Extended\models\evaluation\ai_assisted_relevance_judgments.csv
Review table saved: D:\Course_recomendation_Deep_Learning_Extended\models\evaluation\ai_assisted_relevance_review_table.csv
Queries covered: 37
Judgments rows: 666
Baseline results saved: D:\Course_recomendation_Deep_Learning_Extended\models\evaluation\baseline_results.csv
Baseline summary:
{'queries_evaluated': 37, 'precision_at_5': 1.0, 'recall_at_10': 0.3646304675716441, 'mrr': 1.0, 'ndcg_at_10': 0.9610573091978746}


In [9]:
sample_query = "beginner Python programming"
sample_recommendations = recommend_baseline(
    sample_query,
    top_n=6,
)

print("Sample query:", sample_query)
print(
    sample_recommendations[
        [
            "course_id",
            "course_name",
            "similarity_score",
            "final_score",
        ]
    ].to_string(index=False)
)

print("\nSanity check complete: recommendations were computed without building a full pairwise similarity matrix.")

Sample query: beginner Python programming
course_id                                            course_name  similarity_score  final_score
    21304                       Python Programming for Begineers          0.950845     0.808219
    20980                          Python Programming in 5 Hours          0.761232     0.647047
    19709                        Python Programming For Everyone          0.705965     0.600070
    22170 The Python Programming For Everyone Immersive Training          0.640016     0.544014
    20238                                Programming with Python          0.635805     0.540434
    18965       2022-Master in Core Python Programming in 99Days          0.614230     0.522096

Sanity check complete: recommendations were computed without building a full pairwise similarity matrix.


## Versioned Retrieval Experiments

This section builds and compares three retrieval variants against the AI-assisted relevance judgments:

- Version 1: frozen word-level TF-IDF baseline
- Version 2: separate field TF-IDF with explicit field weights
- Version 3: word TF-IDF plus character n-grams
- Version 3 reranked: top-100 NLP candidates reranked with popularity and data quality

In [10]:
from time import perf_counter
import importlib.metadata as importlib_metadata

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

version_root = MODELS_DIR / "v2"
version_root.mkdir(parents=True, exist_ok=True)

FIELD_WEIGHTS_V2 = {
    "course_name": 3.0,
    "skills": 3.0,
    "subject": 2.0,
    "description": 1.5,
    "metadata": 0.5,
}

CHAR_MIXTURES = [
    (1.00, 0.00),
    (0.90, 0.10),
    (0.80, 0.20),
    (0.70, 0.30),
]

RERANK_WEIGHT_CANDIDATES = [
    (1.00, 0.00, 0.00),
    (0.95, 0.03, 0.02),
    (0.90, 0.07, 0.03),
    (0.85, 0.10, 0.05),
    (0.80, 0.15, 0.05),
]


def normalize_text_series(series: pd.Series) -> pd.Series:
    return (
        series
        .fillna("")
        .astype(str)
        .apply(clean_nlp_text)
    )


def matrix_memory_mb(matrix) -> float:
    total_bytes = 0
    for attribute_name in ["data", "indices", "indptr"]:
        if hasattr(matrix, attribute_name):
            total_bytes += getattr(matrix, attribute_name).nbytes
    return total_bytes / (1024 ** 2)


def normalize_duplicate_keys(dataframe: pd.DataFrame) -> pd.Series:
    return (
        dataframe["course_name"].fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
        + "|" + dataframe["provider"].fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
        + "|" + dataframe["organization"].fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
        + "|" + dataframe["url"].fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
    )


def dedupe_strong_duplicates(dataframe: pd.DataFrame) -> pd.DataFrame:
    if dataframe.empty:
        return dataframe
    working = dataframe.copy()
    working["_duplicate_key"] = normalize_duplicate_keys(working)
    working = working.sort_values(
        by=["final_score", "similarity_score"],
        ascending=False,
    )
    working = working.drop_duplicates(subset=["_duplicate_key"], keep="first")
    return working.drop(columns=["_duplicate_key"]).reset_index(drop=True)


def get_judgment_lookup(query: str) -> dict[str, int]:
    query_judgments = ai_assisted_relevance_judgments[
        ai_assisted_relevance_judgments["query"].eq(query)
    ].copy()
    query_judgments["course_id"] = query_judgments["course_id"].astype(str)
    return query_judgments.set_index("course_id")["relevance_grade"].to_dict()


def evaluate_ranked_frame(query: str, recommendations: pd.DataFrame) -> pd.DataFrame:
    lookup = get_judgment_lookup(query)
    if not lookup:
        warnings.warn(f"No AI-assisted relevance judgments found for query: {query}")
        return pd.DataFrame()

    recommendations = recommendations.copy()
    recommendations["course_id"] = recommendations["course_id"].astype(str)
    score_column = "final_score" if "final_score" in recommendations.columns else "similarity_score"
    recommended_ids = recommendations["course_id"].tolist()
    recommended_scores = recommendations[score_column].astype(float).tolist()
    relevance_grades = [int(lookup.get(course_id, 0)) for course_id in recommended_ids]
    total_relevant = int(sum(grade > 0 for grade in lookup.values()))
    if total_relevant <= 0:
        warnings.warn(f"Empty relevant set for query: {query}")
        total_relevant = 0

    return pd.DataFrame(
        [
            {
                "query": query,
                "precision_at_5": precision_at_k(relevance_grades, 5),
                "recall_at_10": recall_at_k(relevance_grades, 10, total_relevant),
                "reciprocal_rank": reciprocal_rank(relevance_grades),
                "ndcg_at_10": ndcg_at_k(relevance_grades, recommended_scores, 10),
                "number_of_relevant_courses": total_relevant,
                "number_of_retrieved_courses": len(recommended_ids),
            }
        ]
    )


# ---------- Version 2: separate field TF-IDF ----------
v2_start = perf_counter()

v2_frames = {
    "course_name": normalize_text_series(baseline_courses["course_name"]),
    "skills": normalize_text_series(baseline_courses["skills"]),
    "subject": normalize_text_series(baseline_courses["subject"]),
    "description": normalize_text_series(baseline_courses["description"]),
    "metadata": normalize_text_series(
        baseline_courses[["level", "course_type", "certificate_type", "organization", "provider"]]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    ),
}

v2_vectorizers = {
    field_name: TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=max_features,
        stop_words="english",
        sublinear_tf=True,
        dtype=np.float32,
    )
    for field_name, max_features in [
        ("course_name", 15_000),
        ("skills", 20_000),
        ("subject", 10_000),
        ("description", 30_000),
        ("metadata", 10_000),
    ]
}

v2_matrices = {
    field_name: vectorizer.fit_transform(v2_frames[field_name])
    for field_name, vectorizer in v2_vectorizers.items()
}

v2_word_matrix = hstack(
    [
        FIELD_WEIGHTS_V2["course_name"] * v2_matrices["course_name"],
        FIELD_WEIGHTS_V2["skills"] * v2_matrices["skills"],
        FIELD_WEIGHTS_V2["subject"] * v2_matrices["subject"],
        FIELD_WEIGHTS_V2["description"] * v2_matrices["description"],
        FIELD_WEIGHTS_V2["metadata"] * v2_matrices["metadata"],
    ]
).tocsr()

v2_training_time = perf_counter() - v2_start


def transform_v2_query(query: str):
    cleaned_query = clean_nlp_text(query)
    parts = [
        FIELD_WEIGHTS_V2["course_name"] * v2_vectorizers["course_name"].transform([cleaned_query]),
        FIELD_WEIGHTS_V2["skills"] * v2_vectorizers["skills"].transform([cleaned_query]),
        FIELD_WEIGHTS_V2["subject"] * v2_vectorizers["subject"].transform([cleaned_query]),
        FIELD_WEIGHTS_V2["description"] * v2_vectorizers["description"].transform([cleaned_query]),
        FIELD_WEIGHTS_V2["metadata"] * v2_vectorizers["metadata"].transform([cleaned_query]),
    ]
    return hstack(parts).tocsr()


def recommend_v2(query: str, top_n: int = 10, candidate_pool_size: int = 100, rerank_weights=(0.85, 0.10, 0.05), dedupe=True) -> pd.DataFrame:
    query_matrix = transform_v2_query(query)
    similarity_scores = linear_kernel(query_matrix, v2_word_matrix).ravel()
    candidate_indices = np.argsort(-similarity_scores)[:candidate_pool_size]
    candidates = baseline_courses.iloc[candidate_indices].copy()
    candidates["similarity_score"] = similarity_scores[candidate_indices]
    similarity_weight, popularity_weight, quality_weight = rerank_weights
    candidates["final_score"] = (
        similarity_weight * candidates["similarity_score"]
        + popularity_weight * candidates["popularity_score"]
        + quality_weight * candidates["data_quality_score"]
    )
    candidates = candidates[candidates["similarity_score"] > 0].copy()
    if dedupe:
        candidates = dedupe_strong_duplicates(candidates)
    return candidates.sort_values(
        by=["final_score", "similarity_score"],
        ascending=False,
    ).head(top_n).reset_index(drop=True)


# ---------- Version 3: word + character TF-IDF ----------
v3_start = perf_counter()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30_000,
    sublinear_tf=True,
    dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(baseline_courses["_combined_text_clean"])
v3_training_time = perf_counter() - v3_start


def score_v3_query(query: str, word_weight: float, char_weight: float):
    cleaned_query = clean_nlp_text(query)
    word_query = transform_v2_query(cleaned_query)
    char_query = char_vectorizer.transform([cleaned_query])
    word_scores = linear_kernel(word_query, v2_word_matrix).ravel()
    char_scores = linear_kernel(char_query, char_matrix).ravel()
    return word_weight * word_scores + char_weight * char_scores


def recommend_v3(query: str, top_n: int = 10, word_weight: float = 0.80, char_weight: float = 0.20, candidate_pool_size: int = 100, rerank_weights=(0.85, 0.10, 0.05), dedupe=True) -> pd.DataFrame:
    combined_scores = score_v3_query(query, word_weight, char_weight)
    candidate_indices = np.argsort(-combined_scores)[:candidate_pool_size]
    candidates = baseline_courses.iloc[candidate_indices].copy()
    candidates["similarity_score"] = combined_scores[candidate_indices]
    similarity_weight, popularity_weight, quality_weight = rerank_weights
    candidates["final_score"] = (
        similarity_weight * candidates["similarity_score"]
        + popularity_weight * candidates["popularity_score"]
        + quality_weight * candidates["data_quality_score"]
    )
    candidates = candidates[candidates["similarity_score"] > 0].copy()
    if dedupe:
        candidates = dedupe_strong_duplicates(candidates)
    return candidates.sort_values(
        by=["final_score", "similarity_score"],
        ascending=False,
    ).head(top_n).reset_index(drop=True)


# ---------- Evaluation ----------
def evaluate_model(model_version: str, recommend_fn, queries: list[str]):
    per_query_rows = []
    timings = []
    for query in queries:
        start_time = perf_counter()
        recommendations = recommend_fn(query, top_n=10)
        timings.append((perf_counter() - start_time) * 1000.0)
        per_query_df = evaluate_ranked_frame(query, recommendations)
        if per_query_df.empty:
            continue
        per_query_df["model_version"] = model_version
        per_query_rows.append(per_query_df)

    if not per_query_rows:
        return pd.DataFrame(), {
            "model_version": model_version,
            "precision_at_5": 0.0,
            "recall_at_10": 0.0,
            "mrr": 0.0,
            "ndcg_at_10": 0.0,
            "training_time_seconds": 0.0,
            "average_query_time_ms": 0.0,
            "matrix_shape": "(0, 0)",
            "matrix_memory_mb": 0.0,
        }

    per_query_results = pd.concat(per_query_rows, ignore_index=True)
    summary = {
        "model_version": model_version,
        "precision_at_5": float(per_query_results["precision_at_5"].mean()),
        "recall_at_10": float(per_query_results["recall_at_10"].mean()),
        "mrr": float(per_query_results["reciprocal_rank"].mean()),
        "ndcg_at_10": float(per_query_results["ndcg_at_10"].mean()),
        "average_query_time_ms": float(np.mean(timings)),
    }
    return per_query_results, summary


comparison_rows = []
version_outputs = {}

baseline_results_v1, summary_v1 = evaluate_model(
    "word_tfidf_baseline",
    recommend_baseline,
    validation_queries,
)
summary_v1.update(
    {
        "training_time_seconds": 0.0,
        "matrix_shape": str(baseline_matrix.shape),
        "matrix_memory_mb": float(matrix_memory_mb(baseline_matrix)),
    }
)
comparison_rows.append(summary_v1)
version_outputs["word_tfidf_baseline"] = baseline_results_v1

baseline_results_v1.to_csv(EVAL_DIR / "baseline_results.csv", index=False)

v2_results, summary_v2 = evaluate_model(
    "separate_field_tfidf",
    lambda query, top_n=10: recommend_v2(query, top_n=top_n, rerank_weights=(0.85, 0.10, 0.05)),
    validation_queries,
)
summary_v2.update(
    {
        "training_time_seconds": float(v2_training_time),
        "matrix_shape": str(v2_word_matrix.shape),
        "matrix_memory_mb": float(matrix_memory_mb(v2_word_matrix)),
    }
)
comparison_rows.append(summary_v2)
version_outputs["separate_field_tfidf"] = v2_results

v3_mixture_results = []
for word_weight, char_weight in CHAR_MIXTURES:
    version_name = f"word_char_tfidf_{word_weight:.2f}_{char_weight:.2f}"
    results_df, summary = evaluate_model(
        version_name,
        lambda query, top_n=10, ww=word_weight, cw=char_weight: recommend_v3(
            query,
            top_n=top_n,
            word_weight=ww,
            char_weight=cw,
            rerank_weights=(0.85, 0.10, 0.05),
        ),
        validation_queries,
    )
    summary.update(
        {
            "training_time_seconds": float(v2_training_time + v3_training_time),
            "matrix_shape": str(v2_word_matrix.shape),
            "matrix_memory_mb": float(matrix_memory_mb(v2_word_matrix) + matrix_memory_mb(char_matrix)),
        }
    )
    v3_mixture_results.append((version_name, results_df, summary, word_weight, char_weight))

best_v3_mixture = max(v3_mixture_results, key=lambda item: item[2]["ndcg_at_10"])
comparison_rows.append(best_v3_mixture[2])
version_outputs[best_v3_mixture[0]] = best_v3_mixture[1]

reranked_candidates = []
for similarity_weight, popularity_weight, quality_weight in RERANK_WEIGHT_CANDIDATES:
    version_name = f"word_char_tfidf_rerank_{similarity_weight:.2f}_{popularity_weight:.2f}_{quality_weight:.2f}"
    results_df, summary = evaluate_model(
        version_name,
        lambda query, top_n=10, ww=best_v3_mixture[3], cw=best_v3_mixture[4], weights=(similarity_weight, popularity_weight, quality_weight): recommend_v3(
            query,
            top_n=top_n,
            word_weight=ww,
            char_weight=cw,
            rerank_weights=weights,
        ),
        validation_queries,
    )
    summary.update(
        {
            "training_time_seconds": float(v2_training_time + v3_training_time),
            "matrix_shape": str(v2_word_matrix.shape),
            "matrix_memory_mb": float(matrix_memory_mb(v2_word_matrix) + matrix_memory_mb(char_matrix)),
        }
    )
    reranked_candidates.append((version_name, results_df, summary, similarity_weight, popularity_weight, quality_weight))

best_reranked = max(reranked_candidates, key=lambda item: item[2]["ndcg_at_10"])
comparison_rows.append(best_reranked[2])
version_outputs[best_reranked[0]] = best_reranked[1]

comparison_table = pd.DataFrame(comparison_rows).sort_values(
    by=["ndcg_at_10", "precision_at_5", "average_query_time_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)

comparison_path = EVAL_DIR / "model_comparison.csv"
comparison_table.to_csv(comparison_path, index=False)

best_row = comparison_table.iloc[0].to_dict()
selected_version = best_row["model_version"]

print("Model comparison saved:", comparison_path)
print(comparison_table.to_string(index=False))
print("\nSelected version:", selected_version)

# ---------- Save the best model bundle ----------
selected_version_key = selected_version
selected_results = version_outputs[selected_version_key]

model_config = {
    "selected_model_version": selected_version_key,
    "field_weights": FIELD_WEIGHTS_V2,
    "rerank_weights": {
        "similarity_weight": float(best_row["model_version"].startswith("word_char_tfidf_rerank_") and best_reranked[3] or 0.85),
        "popularity_weight": float(best_row["model_version"].startswith("word_char_tfidf_rerank_") and best_reranked[4] or 0.10),
        "quality_weight": float(best_row["model_version"].startswith("word_char_tfidf_rerank_") and best_reranked[5] or 0.05),
    },
    "number_of_courses": int(len(baseline_courses)),
    "number_of_features": int(v2_word_matrix.shape[1]),
    "evaluation_metrics": {
        "precision_at_5": float(best_row["precision_at_5"]),
        "recall_at_10": float(best_row["recall_at_10"]),
        "mrr": float(best_row["mrr"]),
        "ndcg_at_10": float(best_row["ndcg_at_10"]),
    },
    "creation_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "required_packages": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": importlib_metadata.version("scikit-learn"),
        "scipy": importlib_metadata.version("scipy"),
        "joblib": importlib_metadata.version("joblib"),
    },
    "versioned_artifacts": {
        "field_vectorizers": "field_vectorizers.joblib",
        "word_matrix": "word_matrix.joblib",
        "char_vectorizer": "char_vectorizer.joblib",
        "char_matrix": "char_matrix.joblib",
    },
}

with open(version_root / "model_config.json", "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

joblib.dump(
    {
        "course_name": v2_vectorizers["course_name"],
        "skills": v2_vectorizers["skills"],
        "subject": v2_vectorizers["subject"],
        "description": v2_vectorizers["description"],
        "metadata": v2_vectorizers["metadata"],
    },
    version_root / "field_vectorizers.joblib",
)
joblib.dump(v2_word_matrix, version_root / "word_matrix.joblib")
joblib.dump(char_vectorizer, version_root / "char_vectorizer.joblib")
joblib.dump(char_matrix, version_root / "char_matrix.joblib")
joblib.dump(baseline_courses.drop(columns=["_combined_text", "_combined_text_clean"], errors="ignore"), version_root / "courses.joblib")
comparison_table.to_csv(version_root / "evaluation_results.csv", index=False)

# Compatibility artifacts for the Flask loader.
joblib.dump(v2_vectorizers, version_root / "tfidf_vectorizer.joblib")
joblib.dump(v2_word_matrix, version_root / "tfidf_matrix.joblib")

print("Best model bundle saved to:", version_root)
print((version_root / "model_config.json").resolve())


Model comparison saved: D:\Course_recomendation_Deep_Learning_Extended\models\evaluation\model_comparison.csv
                        model_version  precision_at_5  recall_at_10      mrr  ndcg_at_10  average_query_time_ms  training_time_seconds   matrix_shape  matrix_memory_mb
                  word_tfidf_baseline             1.0      0.364630 1.000000    0.961057             120.273441               0.000000 (24646, 50000)         24.058338
word_char_tfidf_rerank_1.00_0.00_0.00             0.4      0.185257 0.683859    0.717189             670.538773              54.773562 (24646, 67582)        255.466888
            word_char_tfidf_0.90_0.10             0.4      0.185257 0.683859    0.717189             757.442800              54.773562 (24646, 67582)        255.466888
                 separate_field_tfidf             0.4      0.185257 0.683859    0.715050              57.029157              17.669239 (24646, 67582)         24.566494

Selected version: word_tfidf_baseline
Best model 

In [11]:
# Rebuild the full comparison table so every evaluated configuration is retained.
full_comparison_rows = [summary_v1, summary_v2]
full_comparison_rows.extend(item[2] for item in v3_mixture_results)
full_comparison_rows.extend(item[2] for item in reranked_candidates)

full_comparison_table = pd.DataFrame(full_comparison_rows).sort_values(
    by=["ndcg_at_10", "precision_at_5", "average_query_time_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)

full_comparison_path = EVAL_DIR / "model_comparison.csv"
full_comparison_table.to_csv(full_comparison_path, index=False)
version_root = MODELS_DIR / "v2"
version_root.mkdir(parents=True, exist_ok=True)

best_row = full_comparison_table.iloc[0].to_dict()
selected_version = best_row["model_version"]

if selected_version == "word_tfidf_baseline":
    selected_rerank_weights = {
        "similarity_weight": baseline_config["similarity_weight"],
        "popularity_weight": baseline_config["popularity_weight"],
        "quality_weight": baseline_config["quality_weight"],
    }
    selected_vectorizer = baseline_vectorizer
    selected_matrix = baseline_matrix
    selected_courses = baseline_courses.drop(columns=["_combined_text", "_combined_text_clean"], errors="ignore").copy()
    selected_courses["course_id"] = selected_courses["course_id"].astype(str)
    joblib.dump(selected_vectorizer, version_root / "tfidf_vectorizer.joblib")
    joblib.dump(selected_matrix, version_root / "tfidf_matrix.joblib")
    joblib.dump(selected_courses, version_root / "courses.joblib")
elif selected_version == "separate_field_tfidf":
    selected_rerank_weights = {
        "similarity_weight": 0.85,
        "popularity_weight": 0.10,
        "quality_weight": 0.05,
    }
    selected_courses = baseline_courses.drop(columns=["_combined_text", "_combined_text_clean"], errors="ignore").copy()
    selected_courses["course_id"] = selected_courses["course_id"].astype(str)
    joblib.dump(v2_vectorizers, version_root / "field_vectorizers.joblib")
    joblib.dump(v2_word_matrix, version_root / "word_matrix.joblib")
    joblib.dump(v2_vectorizers, version_root / "tfidf_vectorizer.joblib")
    joblib.dump(v2_word_matrix, version_root / "tfidf_matrix.joblib")
    joblib.dump(selected_courses, version_root / "courses.joblib")
elif selected_version.startswith("word_char_tfidf"):
    if selected_version.startswith("word_char_tfidf_rerank_"):
        selected_rerank_weights = {
            "similarity_weight": float(best_reranked[3]),
            "popularity_weight": float(best_reranked[4]),
            "quality_weight": float(best_reranked[5]),
        }
    else:
        selected_rerank_weights = {
            "similarity_weight": 0.85,
            "popularity_weight": 0.10,
            "quality_weight": 0.05,
        }
    selected_courses = baseline_courses.drop(columns=["_combined_text", "_combined_text_clean"], errors="ignore").copy()
    selected_courses["course_id"] = selected_courses["course_id"].astype(str)
    joblib.dump(v2_vectorizers, version_root / "field_vectorizers.joblib")
    joblib.dump(v2_word_matrix, version_root / "word_matrix.joblib")
    joblib.dump(char_vectorizer, version_root / "char_vectorizer.joblib")
    joblib.dump(char_matrix, version_root / "char_matrix.joblib")
    joblib.dump(v2_vectorizers, version_root / "tfidf_vectorizer.joblib")
    joblib.dump(v2_word_matrix, version_root / "tfidf_matrix.joblib")
    joblib.dump(selected_courses, version_root / "courses.joblib")
else:
    selected_rerank_weights = {
        "similarity_weight": 0.85,
        "popularity_weight": 0.10,
        "quality_weight": 0.05,
    }
    selected_courses = baseline_courses.drop(columns=["_combined_text", "_combined_text_clean"], errors="ignore").copy()
    selected_courses["course_id"] = selected_courses["course_id"].astype(str)
    joblib.dump(selected_courses, version_root / "courses.joblib")

model_config = {
    "selected_model_version": selected_version,
    "field_weights": FIELD_WEIGHTS_V2,
    "rerank_weights": selected_rerank_weights,
    "number_of_courses": int(len(baseline_courses)),
    "number_of_features": int(selected_matrix.shape[1]) if selected_version == "word_tfidf_baseline" else int(v2_word_matrix.shape[1]),
    "evaluation_metrics": {
        "precision_at_5": float(best_row["precision_at_5"]),
        "recall_at_10": float(best_row["recall_at_10"]),
        "mrr": float(best_row["mrr"]),
        "ndcg_at_10": float(best_row["ndcg_at_10"]),
    },
    "creation_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "required_packages": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": importlib_metadata.version("scikit-learn"),
        "scipy": importlib_metadata.version("scipy"),
        "joblib": importlib_metadata.version("joblib"),
    },
}

with open(version_root / "model_config.json", "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

full_comparison_table.to_csv(version_root / "evaluation_results.csv", index=False)

print("Full model comparison saved:", full_comparison_path)
print(full_comparison_table.to_string(index=False))
print("\nWinning configuration:", selected_version)
print("Saved selected model bundle in:", version_root)


Full model comparison saved: D:\Course_recomendation_Deep_Learning_Extended\models\evaluation\model_comparison.csv
                        model_version  precision_at_5  recall_at_10      mrr  ndcg_at_10  average_query_time_ms  training_time_seconds   matrix_shape  matrix_memory_mb
                  word_tfidf_baseline             1.0      0.364630 1.000000    0.961057             120.273441               0.000000 (24646, 50000)         24.058338
word_char_tfidf_rerank_0.80_0.15_0.05             0.4      0.185257 0.683859    0.717189             664.886127              54.773562 (24646, 67582)        255.466888
word_char_tfidf_rerank_0.95_0.03_0.02             0.4      0.185257 0.683859    0.717189             665.436792              54.773562 (24646, 67582)        255.466888
word_char_tfidf_rerank_1.00_0.00_0.00             0.4      0.185257 0.683859    0.717189             670.538773              54.773562 (24646, 67582)        255.466888
word_char_tfidf_rerank_0.85_0.10_0.05        